In [1]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Dropout, Conv2DTranspose, Concatenate, Add, BatchNormalization, Activation
import os
import glob
import tifffile as tiff
# import numpy as np
# import os
# import matplotlib.pyplot as plt
# from sklearn.metrics import precision_score, recall_score, f1_score
from PIL import Image
import rasterio
from sklearn.model_selection import train_test_split;
import re
import tensorflow as tf
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score

In [2]:
# ---- Reusable building blocks (adapted to support dropout) ----
def conv_block(x, filters, batchnorm=True):
    conv1 = Conv2D(filters, (3, 3), kernel_initializer='he_normal', padding='same')(x)
    if batchnorm:
        conv1 = BatchNormalization(axis=3)(conv1)
    conv1 = Activation('relu')(conv1)

    conv2 = Conv2D(filters, (3, 3), kernel_initializer='he_normal', padding='same')(conv1)
    if batchnorm:
        conv2 = BatchNormalization(axis=3)(conv2)
    conv2 = Activation('relu')(conv2)

    return conv2

def dense_block(inputs, num_filters, dropout_rate=0.0):
    """
    Dense-like block that concatenates input with conv_block output.
    dropout_rate: applied after concatenation (stochastic behavior used for Bayesian approx).
    """
    conv1 = conv_block(inputs, num_filters)
    concat = Concatenate()([inputs, conv1])
    if dropout_rate and dropout_rate > 0.0:
        concat = Dropout(dropout_rate)(concat)  # respects training flag when model called with training=True
    return concat

def residual_conv_block(x, filters, batchnorm=True, dropout_rate=0.0):
    conv1 = Conv2D(filters, (3, 3), kernel_initializer='he_normal', padding='same')(x)
    if batchnorm:
        conv1 = BatchNormalization(axis=3)(conv1)
    conv1 = Activation('relu')(conv1)

    conv2 = Conv2D(filters, (3, 3), kernel_initializer='he_normal', padding='same')(conv1)
    if batchnorm:
        conv2 = BatchNormalization(axis=3)(conv2)
    conv2 = Activation('relu')(conv2)

    # skip/shortcut
    shortcut = Conv2D(filters, kernel_size=(1, 1), kernel_initializer='he_normal', padding='same')(x)
    if batchnorm:
        shortcut = BatchNormalization(axis=3)(shortcut)
    # no activation on shortcut before add (original had Activation on it; kept minimal)
    respath = Add()([shortcut, conv2])

    if dropout_rate and dropout_rate > 0.0:
        respath = Dropout(dropout_rate)(respath)

    respath = Activation('relu')(respath)
    return respath

# ---- Bayesian U-Net builder ----
def bayesian_dense_unet(input_shape, base_filters=64, dropout_rate=0.3):
    """
    Builds a U-Net variant with multiple Dropout layers for MC Dropout (epistemic uncertainty).
    dropout_rate: dropout probability used across the network (applied where appropriate).
    """
    inputs = Input(input_shape)

    # Encoder (dense blocks with dropout)
    conv1 = dense_block(inputs, base_filters, dropout_rate=dropout_rate)
    pool1 = MaxPooling2D(pool_size=(2, 2))(conv1)

    conv2 = dense_block(pool1, base_filters * 2, dropout_rate=dropout_rate)
    pool2 = MaxPooling2D(pool_size=(2, 2))(conv2)

    conv3 = dense_block(pool2, base_filters * 4, dropout_rate=dropout_rate)
    pool3 = MaxPooling2D(pool_size=(2, 2))(conv3)

    conv4 = dense_block(pool3, base_filters * 8, dropout_rate=dropout_rate)
    pool4 = MaxPooling2D(pool_size=(2, 2))(conv4)

    # Bottleneck (two convs + dropout)
    conv5 = Conv2D(base_filters * 16, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(pool4)
    conv5 = Conv2D(base_filters * 16, (3, 3), kernel_initializer='he_normal', padding='same')(conv5)
    conv5 = Activation('relu')(conv5)
    drop5 = Dropout(dropout_rate)(conv5)  # core stochastic layer

    # Decoder (transpose convs + residual blocks + dropout)
    up6 = Conv2DTranspose(base_filters * 8, (2, 2), strides=(2, 2), padding='same')(drop5)
    up6 = Concatenate()([up6, conv4])
    conv6 = dense_block(up6, base_filters * 8, dropout_rate=dropout_rate)

    up7 = Conv2DTranspose(base_filters * 4, (2, 2), strides=(2, 2), padding='same')(conv6)
    up7 = Concatenate()([up7, conv3])
    conv7 = dense_block(up7, base_filters * 4, dropout_rate=dropout_rate)

    up8 = Conv2DTranspose(base_filters * 2, (2, 2), strides=(2, 2), padding='same')(conv7)
    up8 = Concatenate()([up8, conv2])
    conv8 = dense_block(up8, base_filters * 2, dropout_rate=dropout_rate)

    up9 = Conv2DTranspose(base_filters, (2, 2), strides=(2, 2), padding='same')(conv8)
    up9 = Concatenate()([up9, conv1])
    conv9 = dense_block(up9, base_filters, dropout_rate=dropout_rate)

    # Output - binary segmentation (sigmoid)
    outputs = Conv2D(1, (1, 1), activation='sigmoid')(conv9)

    model = Model(inputs=inputs, outputs=outputs)
    return model

def bayesian_residual_unet(input_shape, base_filters=64, dropout_rate=0.3):
    inputs = Input(input_shape)
    
    # Encoder
    conv1 = residual_conv_block(inputs, base_filters, dropout_rate=dropout_rate)
    pool1 = MaxPooling2D(pool_size=(2, 2))(conv1)
    conv2 = residual_conv_block(pool1, base_filters * 2, dropout_rate=dropout_rate)
    pool2 = MaxPooling2D(pool_size=(2, 2))(conv2)
    conv3 = residual_conv_block(pool2, base_filters * 4, dropout_rate=dropout_rate)
    pool3 = MaxPooling2D(pool_size=(2, 2))(conv3)
    conv4 = residual_conv_block(pool3, base_filters * 8, dropout_rate=dropout_rate)
    pool4 = MaxPooling2D(pool_size=(2, 2))(conv4)
    
    # Bottleneck
    conv5 = Conv2D(base_filters * 16, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(pool4)
    conv5 = Conv2D(base_filters * 16, (3, 3), kernel_initializer='he_normal', padding='same')(conv5)
    conv5 = Activation('relu')(conv5)
    drop5 = Dropout(dropout_rate)(conv5)
    
    # Decoder
    up6 = Conv2DTranspose(base_filters * 8, (2, 2), strides=(2, 2), padding='same')(drop5)
    up6 = Concatenate()([up6, conv4])
    conv6 = residual_conv_block(up6, base_filters * 8, dropout_rate=dropout_rate)
    up7 = Conv2DTranspose(base_filters * 4, (2, 2), strides=(2, 2), padding='same')(conv6)
    up7 = Concatenate()([up7, conv3])
    conv7 = residual_conv_block(up7, base_filters * 4, dropout_rate=dropout_rate)
    up8 = Conv2DTranspose(base_filters * 2, (2, 2), strides=(2, 2), padding='same')(conv7)
    up8 = Concatenate()([up8, conv2])
    conv8 = residual_conv_block(up8, base_filters * 2, dropout_rate=dropout_rate)
    up9 = Conv2DTranspose(base_filters, (2, 2), strides=(2, 2), padding='same')(conv8)
    up9 = Concatenate()([up9, conv1])
    conv9 = residual_conv_block(up9, base_filters, dropout_rate=dropout_rate)
    
    # Output
    outputs = Conv2D(1, (1, 1), activation='sigmoid')(conv9)
    
    model = Model(inputs=inputs, outputs=outputs)
    return model

In [3]:
def load_data_new(input_dir, mask_dir, corrupt=True):

    # Ensure the directories exist
    if not os.path.exists(input_dir):
        print(f"Input directory {input_dir} does not exist.")
        return None, None
    
    if not os.path.exists(mask_dir):
        print(f"Mask directory {mask_dir} does not exist.")
        return None, None

    # Search for .tif files in the directories
    input_files = glob.glob(os.path.join(input_dir, '*.tif'))
    mask_files = glob.glob(os.path.join(mask_dir, '*.tif'))

    # print("Found input files:", input_files)
    # print("Found mask files:", mask_files)

    if not input_files:
        print(f"No input files found in {input_dir}.")
        return None, None

    if not mask_files:
        print(f"No mask files found in {mask_dir}.")
        return None, None

    images = []
    masks = []

    for mask_file in mask_files:
        # Extract the number i from the mask filename
        if corrupt:
            match = re.search(r'NDWI_Mask_(\d+)_resized_corrupt.tif', os.path.basename(mask_file))
        else:
            match = re.search(r'NDWI_Mask_(\d+)_resized.tif', os.path.basename(mask_file))
            
        if match:
            i = match.group(1)
            input_file = os.path.join(input_dir, f'{i}.tif')
            # print("Input File : ",input_file)
            # Check if the corresponding input file exists
            if os.path.exists(input_file):
                # Read input file
                with rasterio.open(input_file) as src:
                    img = src.read(1)  # Read the first band assuming it's a single-band image
                    images.append(img)

                # Read mask file
                with rasterio.open(mask_file) as src:
                    msk = src.read(1)  # Read the first band assuming it's a single-band image
                    masks.append(msk)
            else:
                print(f"Corresponding input file {input_file} for mask {mask_file} not found.")

    if not images or not masks:
        print("No matching pairs of images and masks found.")
        return None, None

    return np.array(images), np.array(masks)

In [4]:
def dice_coefficient(y_true, y_pred, smooth=1):
    y_true_f = y_true.flatten()
    y_pred_f = y_pred.flatten()
    intersection = np.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (np.sum(y_true_f) + np.sum(y_pred_f) + smooth)

def iou(y_true, y_pred, smooth=1):
    y_true_f = y_true.flatten()
    y_pred_f = y_pred.flatten()
    intersection = np.sum(y_true_f * y_pred_f)
    union = np.sum(y_true_f) + np.sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)

def save_image(image, filepath):
    image = (image * 255).astype(np.uint8)  # Scale image to 0-255
    tiff.imwrite(filepath, image)

In [5]:
def predict_scaled_variance(model, mask_dir, image_dir, output_dir, number=-1, mc_samples=20, bayesian=False):
    dice_scores = []
    iou_scores = []
    precisions = []
    recalls = []
    f1_scores = []

    os.makedirs(output_dir, exist_ok=True)

    mask_files = [f for f in os.listdir(mask_dir) if f.endswith('_resized.tif')]
    print(f"Found {len(mask_files)} mask files")

    # -------------------------------------------------------------
    # STEP 1 — Compute global max uncertainty (ONLY if bayesian=True)
    # -------------------------------------------------------------
    global_uncertainty_max = 0
    if bayesian:
        print("Computing global uncertainty max...")

        for mask_file in mask_files:
            i_str = mask_file.split('_')[2]
            image_file = f"{i_str}.tif"

            img_path = os.path.join(image_dir, image_file)
            if not os.path.exists(img_path):
                continue

            img = tiff.imread(img_path)

            if len(img.shape) == 2:
                img = np.expand_dims(img, axis=-1)
            elif img.shape[0] == 2:
                img = img[0]

            preds = []
            for _ in range(mc_samples):
                pred = model(np.expand_dims(img, axis=0), training=True)[0].numpy()
                preds.append(pred)
            preds = np.stack(preds, axis=0)
            unc = np.var(preds, axis=0)

            local_max = unc.max()
            if local_max > global_uncertainty_max:
                global_uncertainty_max = local_max

        print("Global maximum uncertainty value:", global_uncertainty_max)

    # -------------------------------------------------------------

    for i, mask_file in enumerate(mask_files):
        i_str = mask_file.split('_')[2]
        image_file = f"{i_str}.tif"

        mask_path = os.path.join(mask_dir, mask_file)
        image_path = os.path.join(image_dir, image_file)

        # Load image and mask
        arbitrary_img = tiff.imread(image_path)
        arbitrary_mask = tiff.imread(mask_path)

        # Ensure proper shape (512, 512, 1)
        if len(arbitrary_img.shape) == 2:
            arbitrary_img = np.expand_dims(arbitrary_img, axis=-1)
        elif arbitrary_img.shape[0] == 2:
            arbitrary_img = arbitrary_img[0]

        # ---------- Bayesian Inference ----------
        if bayesian:
            preds = []
            for _ in range(mc_samples):
                pred = model(np.expand_dims(arbitrary_img, axis=0), training=True)[0].numpy()
                preds.append(pred)
            preds = np.stack(preds, axis=0)
            predicted_mask = np.mean(preds, axis=0)
            uncertainty_map = np.var(preds, axis=0)

            # Save uncertainty heatmap (NOW uses fixed global scale)
            plt.figure(figsize=(5, 5))
            plt.imshow(
                uncertainty_map.squeeze(),
                cmap='hot',
                vmin=0,
                vmax=global_uncertainty_max  # <<< FIXED SCALE
            )
            plt.colorbar(label='Uncertainty')
            plt.title(f'Uncertainty Heatmap - {i_str}')
            plt.axis('off')
            plt.savefig(f"./{output_dir}/Uncertainty_Heatmap_{i_str}.png", bbox_inches='tight')
            plt.close()

        else:
            predicted_mask = model.predict(np.expand_dims(arbitrary_img, axis=0))[0]

        # ---------- Threshold Prediction ----------
        predicted_mask_thresh = (predicted_mask > 0.5).astype(np.uint8)

        # ---------- Save Predicted Masks ----------
        save_image(predicted_mask, f"./{output_dir}/Predicted_Image_{i_str}.tif")
        save_image(predicted_mask_thresh, f"./{output_dir}/{i_str}.tif")

        # ---------- Visualization ----------
        # plt.figure(figsize=(10, 5))
        # plt.subplot(2, 3, 1)
        # plt.imshow(arbitrary_img.squeeze(), cmap='gray')
        # plt.title('Input Image')
        # plt.axis('off')

        # plt.subplot(2, 3, 2)
        # plt.imshow(arbitrary_mask, cmap='gray')
        # plt.title('Actual Mask')
        # plt.axis('off')

        # plt.subplot(2, 3, 3)
        # plt.imshow(predicted_mask.squeeze(), cmap='gray')
        # plt.title('Predicted Mask (Raw)')
        # plt.axis('off')

        # plt.subplot(2, 3, 4)
        # plt.imshow(predicted_mask_thresh.squeeze(), cmap='gray')
        # plt.title('Predicted Mask (Thresh)')
        # plt.axis('off')

        # if bayesian:
        #     plt.subplot(2, 3, 5)
        #     plt.imshow(
        #         uncertainty_map.squeeze(),
        #         cmap='hot',
        #         vmin=0,
        #         vmax=global_uncertainty_max  # <<< FIXED SCALE
        #     )
        #     plt.title('Uncertainty Heatmap')
        #     plt.axis('off')

        # plt.tight_layout()
        # plt.savefig(f"./{output_dir}/Visualization_{i_str}.jpg", bbox_inches='tight')
        # plt.close()

        # ---------- Metrics ----------
        dice = dice_coefficient(arbitrary_mask, predicted_mask_thresh)
        iou_score = iou(arbitrary_mask, predicted_mask_thresh)
        precision = precision_score(arbitrary_mask.flatten(), predicted_mask_thresh.flatten())
        recall = recall_score(arbitrary_mask.flatten(), predicted_mask_thresh.flatten())
        f1 = f1_score(arbitrary_mask.flatten(), predicted_mask_thresh.flatten())

        dice_scores.append(dice)
        iou_scores.append(iou_score)
        precisions.append(precision)
        recalls.append(recall)
        f1_scores.append(f1)

        if i + 1 == number:
            break

    # ---------- Aggregate Metrics ----------
    mean_dice = np.mean(dice_scores)
    mean_iou = np.mean(iou_scores)
    mean_precision = np.mean(precisions)
    mean_recall = np.mean(recalls)
    mean_f1 = np.mean(f1_scores)

    print(f"Mean Dice Coefficient: {mean_dice:.4f}")
    print(f"Mean IoU: {mean_iou:.4f}")
    print(f"Mean Precision: {mean_precision:.4f}")
    print(f"Mean Recall: {mean_recall:.4f}")
    print(f"Mean F1 Score: {mean_f1:.4f}")

    return mean_dice, mean_iou, mean_precision, mean_recall, mean_f1


In [6]:
def predict_unscaled_variance(model, mask_dir, image_dir, output_dir, number=-1, mc_samples=20, bayesian=False):
    dice_scores = []
    iou_scores = []
    precisions = []
    recalls = []
    f1_scores = []

    os.makedirs(output_dir, exist_ok=True)

    mask_files = [f for f in os.listdir(mask_dir) if f.endswith('_resized.tif')]
    print(f"Found {len(mask_files)} mask files")

    for i, mask_file in enumerate(mask_files):
        i_str = mask_file.split('_')[2]
        image_file = f"{i_str}.tif"

        mask_path = os.path.join(mask_dir, mask_file)
        image_path = os.path.join(image_dir, image_file)

        # Load image and mask
        arbitrary_img = tiff.imread(image_path)
        arbitrary_mask = tiff.imread(mask_path)

        # Ensure proper shape (512, 512, 1)
        if len(arbitrary_img.shape) == 2:
            arbitrary_img = np.expand_dims(arbitrary_img, axis=-1)
        elif arbitrary_img.shape[0] == 2:
            arbitrary_img = arbitrary_img[0]

        # ---------- Bayesian Inference ----------
        if bayesian:
            preds = []
            for _ in range(mc_samples):
                pred = model(np.expand_dims(arbitrary_img, axis=0), training=True)[0].numpy()
                preds.append(pred)
            preds = np.stack(preds, axis=0)
            predicted_mask = np.mean(preds, axis=0)
            uncertainty_map = np.var(preds, axis=0)

            # Save uncertainty heatmap
            plt.figure(figsize=(5, 5))
            plt.imshow(uncertainty_map.squeeze(), cmap='hot')
            plt.colorbar(label='Uncertainty')
            plt.title(f'Uncertainty Heatmap - {i_str}')
            plt.axis('off')
            plt.savefig(f"./{output_dir}/Uncertainty_Heatmap_{i_str}.png", bbox_inches='tight')
            plt.close()

        else:
            predicted_mask = model.predict(np.expand_dims(arbitrary_img, axis=0))[0]

        # ---------- Threshold Prediction ----------
        predicted_mask_thresh = (predicted_mask > 0.5).astype(np.uint8)

        # ---------- Save Predicted Masks ----------
        save_image(predicted_mask, f"./{output_dir}/Predicted_Image_{i_str}.tif")
        save_image(predicted_mask_thresh, f"./{output_dir}/{i_str}.tif")

        # # ---------- Visualization ----------
        # plt.figure(figsize=(10, 5))
        # plt.subplot(2, 3, 1)
        # plt.imshow(arbitrary_img.squeeze(), cmap='gray')
        # plt.title('Input Image')
        # plt.axis('off')

        # plt.subplot(2, 3, 2)
        # plt.imshow(arbitrary_mask, cmap='gray')
        # plt.title('Actual Mask')
        # plt.axis('off')

        # plt.subplot(2, 3, 3)
        # plt.imshow(predicted_mask.squeeze(), cmap='gray')
        # plt.title('Predicted Mask (Raw)')
        # plt.axis('off')

        # plt.subplot(2, 3, 4)
        # plt.imshow(predicted_mask_thresh.squeeze(), cmap='gray')
        # plt.title('Predicted Mask (Thresh)')
        # plt.axis('off')

        # if bayesian:
        #     plt.subplot(2, 3, 5)
        #     plt.imshow(uncertainty_map.squeeze(), cmap='hot')
        #     plt.title('Uncertainty Heatmap')
        #     plt.axis('off')

        # plt.tight_layout()
        # plt.savefig(f"./{output_dir}/Visualization_{i_str}.jpg", bbox_inches='tight')
        # plt.close()

        # ---------- Metrics ----------
        dice = dice_coefficient(arbitrary_mask, predicted_mask_thresh)
        iou_score = iou(arbitrary_mask, predicted_mask_thresh)
        precision = precision_score(arbitrary_mask.flatten(), predicted_mask_thresh.flatten())
        recall = recall_score(arbitrary_mask.flatten(), predicted_mask_thresh.flatten())
        f1 = f1_score(arbitrary_mask.flatten(), predicted_mask_thresh.flatten())

        dice_scores.append(dice)
        iou_scores.append(iou_score)
        precisions.append(precision)
        recalls.append(recall)
        f1_scores.append(f1)

        if i + 1 == number:
            break

    # ---------- Aggregate Metrics ----------
    mean_dice = np.mean(dice_scores)
    mean_iou = np.mean(iou_scores)
    mean_precision = np.mean(precisions)
    mean_recall = np.mean(recalls)
    mean_f1 = np.mean(f1_scores)

    print(f"Mean Dice Coefficient: {mean_dice:.4f}")
    print(f"Mean IoU: {mean_iou:.4f}")
    print(f"Mean Precision: {mean_precision:.4f}")
    print(f"Mean Recall: {mean_recall:.4f}")
    print(f"Mean F1 Score: {mean_f1:.4f}")

    return mean_dice, mean_iou, mean_precision, mean_recall, mean_f1

In [7]:
def predict_entropy(model, mask_dir, image_dir, output_dir,
             number=-1, mc_samples=20, bayesian=False):

    dice_scores = []
    iou_scores = []
    precisions = []
    recalls = []
    f1_scores = []

    os.makedirs(output_dir, exist_ok=True)

    mask_files = [f for f in os.listdir(mask_dir) if f.endswith('_resized.tif')]
    print(f"Found {len(mask_files)} mask files")

    for i, mask_file in enumerate(mask_files):

        i_str = mask_file.split('_')[2]
        image_file = f"{i_str}.tif"

        mask_path = os.path.join(mask_dir, mask_file)
        image_path = os.path.join(image_dir, image_file)

        # Load image & mask
        arbitrary_img = tiff.imread(image_path)
        arbitrary_mask = tiff.imread(mask_path)

        # Ensure 512x512x1 shape
        if len(arbitrary_img.shape) == 2:
            arbitrary_img = np.expand_dims(arbitrary_img, axis=-1)
        elif arbitrary_img.shape[0] == 2:
            arbitrary_img = arbitrary_img[0]

        # ---------- Bayesian Predictive Entropy ----------
        if bayesian:
            preds = []
            for _ in range(mc_samples):
                p = model(np.expand_dims(arbitrary_img, axis=0), training=True)[0].numpy()
                preds.append(p)

            preds = np.stack(preds, axis=0)          # shape: (K, H, W, 1)
            predicted_mask = np.mean(preds, axis=0)  # MC average

            # Compute predictive entropy
            eps = 1e-12
            p = np.clip(predicted_mask, eps, 1 - eps)
            uncertainty_map = -(p * np.log(p) + (1 - p) * np.log(1 - p))

            # Save uncertainty heatmap
            plt.figure(figsize=(5, 5))
            plt.imshow(uncertainty_map.squeeze(), cmap='hot')
            plt.colorbar(label='Predictive Entropy')
            plt.title(f'Uncertainty Heatmap - {i_str}')
            plt.axis('off')
            plt.savefig(f"./{output_dir}/Uncertainty_Heatmap_{i_str}.png",
                        bbox_inches='tight')
            plt.close()

        else:
            predicted_mask = model.predict(np.expand_dims(arbitrary_img, axis=0))[0]

        # ---------- Threshold ----------
        predicted_mask_thresh = (predicted_mask > 0.5).astype(np.uint8)

        # ---------- Save Masks ----------
        save_image(predicted_mask,
                   f"./{output_dir}/Predicted_Image_{i_str}.tif")
        save_image(predicted_mask_thresh,
                   f"./{output_dir}/{i_str}.tif")

        # ----------Visualisation----------
        # plt.figure(figsize=(10, 5))

        # plt.subplot(2, 3, 1)
        # plt.imshow(arbitrary_img.squeeze(), cmap='gray')
        # plt.title('Input Image')
        # plt.axis('off')

        # plt.subplot(2, 3, 2)
        # plt.imshow(arbitrary_mask, cmap='gray')
        # plt.title('Actual Mask')
        # plt.axis('off')

        # plt.subplot(2, 3, 3)
        # plt.imshow(predicted_mask.squeeze(), cmap='gray')
        # plt.title('Predicted Mask (Raw)')
        # plt.axis('off')

        # plt.subplot(2, 3, 4)
        # plt.imshow(predicted_mask_thresh.squeeze(), cmap='gray')
        # plt.title('Predicted Mask (Thresh)')
        # plt.axis('off')

        # if bayesian:
        #     plt.subplot(2, 3, 5)
        #     plt.imshow(uncertainty_map.squeeze(), cmap='hot')
        #     plt.title('Uncertainty Heatmap')
        #     plt.axis('off')

        # plt.tight_layout()
        # plt.savefig(f"./{output_dir}/Visualization_{i_str}.jpg",
        #             bbox_inches='tight')
        # plt.close()

        # ---------- Metrics ----------
        dice = dice_coefficient(arbitrary_mask, predicted_mask_thresh)
        iou_score = iou(arbitrary_mask, predicted_mask_thresh)
        precision = precision_score(arbitrary_mask.flatten(),
                                    predicted_mask_thresh.flatten())
        recall = recall_score(arbitrary_mask.flatten(),
                              predicted_mask_thresh.flatten())
        f1 = f1_score(arbitrary_mask.flatten(),
                      predicted_mask_thresh.flatten())

        dice_scores.append(dice)
        iou_scores.append(iou_score)
        precisions.append(precision)
        recalls.append(recall)
        f1_scores.append(f1)

        if i + 1 == number:
            break

    # ---------- Aggregate Metrics ----------
    print("\n===== METRICS =====")
    print(f"Mean Dice Coefficient: {np.mean(dice_scores):.4f}")
    print(f"Mean IoU: {np.mean(iou_scores):.4f}")
    print(f"Mean Precision: {np.mean(precisions):.4f}")
    print(f"Mean Recall: {np.mean(recalls):.4f}")
    print(f"Mean F1 Score: {np.mean(f1_scores):.4f}")

    return np.mean(dice_scores), np.mean(iou_scores), np.mean(precisions), np.mean(recalls), np.mean(f1_scores)

In [8]:
dropout_rate = 0.3

corruptions = [2]
configs = ['erosion', 'mixed'] # 'dilation'
model_types = ['dense', 'residual']
metrics = ['Entropy']
input_dir = './data_new/'
train = True

for corruption in corruptions:
    for config in configs:
    
        mask_dir = f'./GEE_Masks/GEE_resized/train_gee/train_{corruption}_gee_{config}'
        # Load data
        images, masks = load_data_new(input_dir, mask_dir)

        # Define input shape and number of classes
        images = np.expand_dims(images, axis=-1)
        input_shape = images.shape[1:]
        
        for model_type in model_types:

            model_name = f'Bayesian_UNet_{model_type}_{config}_{corruption}'
            print(model_name)
            
            if model_type == 'dense':
                if train:
                    model = bayesian_dense_unet(input_shape, dropout_rate=dropout_rate)
                    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

                    early_stopping = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

                    # Train the model
                    model.fit(images, masks, validation_split=0.125, epochs=20, batch_size=2)  # Adjust epochs and batch size as needed
                    model.save(model_name)
            
                mask_dir = './GEE_Masks/GEE_resized/test_gee/'
                model = bayesian_dense_unet(input_shape, dropout_rate=dropout_rate)
                model = tf.keras.models.load_model(model_name, compile=False)
                model.compile()
                
                for metric in metrics:

                    output_dir = f'./GEE_Output/Bayesian_UNet/{model_type}/{config}/{corruption}/{metric}/'
                    os.makedirs(output_dir,exist_ok=True)
                    
                    if metric == "Entropy":

                        predict_entropy(
                            model,
                            mask_dir,
                            input_dir,
                            output_dir,
                            mc_samples=20,        # number of MC forward passes
                            bayesian=True         # activate Bayesian inference
                        )
                
            elif model_type == 'residual':
                if train:
                    model = bayesian_residual_unet(input_shape, dropout_rate=dropout_rate)
                    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

                    early_stopping = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

                    # Train the model
                    model.fit(images, masks, validation_split=0.125, epochs=20, batch_size=2)  # Adjust epochs and batch size as needed
                    model.save(model_name)
            
                mask_dir = './GEE_Masks/GEE_resized/test_gee/'
                model = bayesian_residual_unet(input_shape, dropout_rate=dropout_rate)
                model = tf.keras.models.load_model(model_name, compile=False)
                model.compile()
                
                for metric in metrics:

                    output_dir = f'./GEE_Output/Bayesian_UNet/{model_type}/{config}/{corruption}/{metric}/'
                    os.makedirs(output_dir,exist_ok=True)
                    
                    if metric == "Entropy":

                        predict_entropy(
                            model,
                            mask_dir,
                            input_dir,
                            output_dir,
                            mc_samples=20,        # number of MC forward passes
                            bayesian=True         # activate Bayesian inference
                        )

c:\Users\ADMIN\anaconda3\envs\tf_gpu\lib\site-packages\rasterio\__init__.py:368: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)


Bayesian_UNet_dense_erosion_2
Epoch 1/20
442/442 [==============================] - 234s 524ms/step - loss: 0.2218 - accuracy: 0.9323 - val_loss: 0.2644 - val_accuracy: 0.9262
Epoch 2/20
442/442 [==============================] - 241s 546ms/step - loss: 0.1929 - accuracy: 0.9450 - val_loss: 0.1942 - val_accuracy: 0.9445
Epoch 3/20
442/442 [==============================] - 241s 545ms/step - loss: 0.2337 - accuracy: 0.9337 - val_loss: 0.2139 - val_accuracy: 0.9447
Epoch 4/20
442/442 [==============================] - 241s 546ms/step - loss: 0.1786 - accuracy: 0.9487 - val_loss: 0.1750 - val_accuracy: 0.9435
Epoch 5/20
442/442 [==============================] - 241s 546ms/step - loss: 0.1702 - accuracy: 0.9490 - val_loss: 0.1811 - val_accuracy: 0.9463
Epoch 6/20
442/442 [==============================] - 241s 546ms/step - loss: 0.1712 - accuracy: 0.9506 - val_loss: 0.1742 - val_accuracy: 0.9467
Epoch 7/20
442/442 [==============================] - 241s 546ms/step - loss: 0.1688 - accurac

INFO:tensorflow:Assets written to: Bayesian_UNet_dense_erosion_2\assets


INFO:tensorflow:Assets written to: Bayesian_UNet_dense_erosion_2\assets


Found 253 mask files


...

INFO:tensorflow:Assets written to: Bayesian_UNet_residual_mixed_2\assets


INFO:tensorflow:Assets written to: Bayesian_UNet_residual_mixed_2\assets


Found 253 mask files


c:\Users\ADMIN\anaconda3\envs\tf_gpu\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\ADMIN\anaconda3\envs\tf_gpu\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\ADMIN\anaconda3\envs\tf_gpu\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\ADMIN\anaconda3\envs\tf_gpu\lib\site-packag


===== METRICS =====
Mean Dice Coefficient: 0.6671
Mean IoU: 0.6064
Mean Precision: 0.7348
Mean Recall: 0.6647
Mean F1 Score: 0.6636
